# Furas Data Pipeline — Bronze to Silver

## Purpose
This notebook transforms raw tender data from the Bronze layer into clean, standardized, and reliable Silver datasets.

## Input
- Raw tender data stored in Azure Data Lake Storage (ADLS) Bronze layer.
- Data is collected from multiple tender sources with different structures and formats.

## Main Processing
- Read raw data from each source.
- Clean and standardize column names and data types.
- Standardize dates, statuses, tender types, categories, and regions.
- Generate a unique `tender_key` for each tender.
- Map all sources to a common Silver schema.
- Use Delta Lake `MERGE` for incremental source updates.
- Combine the sources into a unified Silver dataset.
- Handle known cross-source duplicate tenders.

## Output
- Source-specific Silver Delta tables.
- One unified and deduplicated Silver tender dataset.

## Technologies
Azure Databricks | PySpark | Delta Lake | ADLS Gen2

## Pipeline Position
Bronze (Raw) → **Silver (Clean & Standardized)** → Gold (Business-Ready)

In [0]:
#checking that Databricks can read the files in bronze layer
#verifies that Databricks can access our Bronze layer in Azure Data Lake

bronze_path = "abfss://bronze@tenderdatalake11.dfs.core.windows.net/"

# list the files:
files = dbutils.fs.ls(bronze_path)

display(files)

In [0]:
# This function handles incremental loading into the Silver layer
# It uses the unique tender key to check whether a tender already exists. Existing tenders are updated, new tenders are inserted, and if the Silver table does  # not exist yet, the function creates it as a Delta table."

from delta.tables import DeltaTable

def merge_to_silver(source_df, silver_path, source_name):
    """
    Incrementally upsert a source DataFrame into its Silver Delta table.
    """

    if DeltaTable.isDeltaTable(spark, silver_path):
        target = DeltaTable.forPath(spark, silver_path)

        (
            target.alias("target")
            .merge(
                source_df.alias("source"),
                "target.tender_key = source.tender_key"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"{source_name}: Silver MERGE completed.")

    else:
        # First-ever load
        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .save(silver_path)
        )

        print(f"{source_name}: Silver table created.")

## Source NO.1 : Nupco

In [0]:
# ============================================================
# STEP: Locate NUPCO Bronze data
# Purpose:
# Define the NUPCO Bronze path and verify that the raw
# ingestion files are available before transformation.
# ============================================================

nupco_path = bronze_path + "nupco/"

display(dbutils.fs.ls(nupco_path))

In [0]:
#Each ingestion is stored in a date-based Bronze folder. This cell automatically selects the latest NUPCO ingestion folder, allowing the pipeline to process  the newest data without manually changing the path.

nupco_folders = dbutils.fs.ls(nupco_path)

latest_nupco_folder = sorted(
    [f.path for f in nupco_folders if f.isDir()],
    reverse=True
)[0]

print("Latest NUPCO folder:", latest_nupco_folder)

display(dbutils.fs.ls(latest_nupco_folder))

In [0]:

# # Get JSON files from the latest NUPCO folder
# nupco_files = [
#     f for f in dbutils.fs.ls(latest_nupco_folder)
#     if not f.isDir() and f.name.endswith(".json")
# ]

# # Select the latest JSON file
# latest_nupco_file = max(
#     nupco_files,
#     key=lambda f: f.modificationTime
# )

# nupco_file = latest_nupco_file.path

# print("Latest NUPCO file:", nupco_file)

# nupco_raw = spark.read.option("multiline", "true").json(nupco_file)

# nupco_raw.printSchema()


# new update for this cell:
# ============================================================
# STEP: Load latest NUPCO Bronze file
# Purpose:
# Find the latest JSON file from the newest NUPCO ingestion
# and load the raw data into a Spark DataFrame.
# ============================================================

nupco_files = [
    f for f in dbutils.fs.ls(latest_nupco_folder)
    if not f.isDir() and f.name.endswith(".json")
]

if not nupco_files:
    raise ValueError(
        f"No JSON files found in {latest_nupco_folder}"
    )

latest_nupco_file = max(
    nupco_files,
    key=lambda f: f.modificationTime
)

nupco_file = latest_nupco_file.path

print("Latest NUPCO file:", nupco_file)

nupco_raw = (
    spark.read
    .option("multiline", "true")
    .json(nupco_file)
)

nupco_raw.printSchema()


In [0]:
#NUPCO stores its tender records inside a nested JSON array called data.
#This cell uses Spark's explode function to convert that array into individual rows, giving us one tender per row while preserving the source and ingestion timestamp.

from pyspark.sql.functions import explode, col

nupco_flat = (
    nupco_raw
    .select(
        explode("data").alias("tender"),
        col("source"),
        col("ingestion_timestamp")
    )
)

display(nupco_flat)

In [0]:
#After flattening the NUPCO tender array, this cell extracts the required fields from each nested tender object.
#It also preserves the source and ingestion timestamp for traceability.

nupco_flat = nupco_flat.select(
    col("tender.tender_id").alias("tender_id"),
    col("tender.tender_name").alias("tender_name"),
    col("tender.tender_type").alias("tender_type"),
    col("tender.submission_deadline").alias("submission_deadline"),
    col("tender.bid_opening").alias("bid_opening"),
    col("tender.tender_url").alias("tender_url"),
    col("source"),
    col("ingestion_timestamp")
)

display(nupco_flat)

In [0]:
nupco_flat.count()

In [0]:
#This cell checks the structure and data types of the extracted NUPCO data before we standardize it into the Silver schema.

nupco_flat.printSchema()

In [0]:
#This validation cell checks the number of extracted NUPCO tenders and displays their current schema before standardization.

print("Number of tenders:", nupco_flat.count())

nupco_flat.printSchema()

In [0]:
#This cell inspects sample NUPCO date and timestamp values so we can understand their raw formats before converting them to standardized Silver data types.

nupco_flat.select(
    "submission_deadline",
    "bid_opening",
    "ingestion_timestamp"
).show(10, truncate=False)

In [0]:
#This cell performs a data-quality check by counting the number of missing values in every extracted NUPCO column.

from pyspark.sql.functions import col, sum as spark_sum

nupco_flat.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in nupco_flat.columns
]).show()

In [0]:
#This cell performs the main NUPCO cleaning and standardization.
#It cleans text fields, converts raw date strings into proper date types, standardizes the source name.
#and generates a unique SHA-256 tender key. The result is a cleaned NUPCO dataset that can then be mapped to the complete common Silver schema.

from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    to_timestamp,
    lit,
    sha2,
    concat_ws,
    regexp_replace
)

nupco_silver = (
    nupco_flat

    # Clean text
    .withColumn("source_tender_id", trim(col("tender_id")))
    .withColumn("tender_name", trim(col("tender_name")))
    .withColumn("tender_type", trim(col("tender_type")))

    # Remove weekday and convert to Date
    # Example: "Thu, 01/10/2026" -> "01/10/2026"
    .withColumn(
        "closing_date",
        to_date(
            regexp_replace(col("submission_deadline"), r"^[A-Za-z]{3},\s*", ""),
            "dd/MM/yyyy"
        )
    )
    .withColumn(
        "opening_date",
        to_date(
            regexp_replace(col("bid_opening"), r"^[A-Za-z]{3},\s*", ""),
            "dd/MM/yyyy"
        )
    )

    # Convert ingestion timestamp
    .withColumn(
        "ingestion_timestamp",
        to_timestamp(col("ingestion_timestamp"))
    )

    # Standard source name
    .withColumn("source", lit("nupco"))

    # Generate unique Furas key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Select Silver columns
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "tender_name",
        "tender_type",
        "closing_date",
        "opening_date",
        "tender_url",
        "ingestion_timestamp"
    )
)

In [0]:
display(nupco_silver)

In [0]:
#This cell checks whether the previous transformation produced a healthy nupco_silver dataset before we continue to the complete Silver schema and write it.

nupco_silver.printSchema()

print(
    "Null closing dates:",
    nupco_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    nupco_silver.filter(col("opening_date").isNull()).count()
)

print("Total rows:", nupco_silver.count())

print(
    "Unique tender keys:",
    nupco_silver.select("tender_key").distinct().count()
)

In [0]:
#This cell defines the standard schema that all tender sources must follow in the Silver layer.
#Because every tender source has a different structure, we created one common Silver schema containing 24 standardized columns.
#Each source is transformed to match this schema before the datasets are combined.
#It standardizes identifiers, dates, statuses, categories, and other tender information while still preserving important original source values.

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    ArrayType,
    DecimalType,
    BooleanType
)

silver_schema = StructType([
    StructField("tender_key", StringType(), True),
    StructField("source", StringType(), False),
    StructField("source_tender_id", StringType(), True),
    StructField("reference_number", StringType(), True),

    StructField("tender_name", StringType(), True),
    StructField("tender_name_en", StringType(), True),
    StructField("description", StringType(), True),
    StructField("agency_name", StringType(), True),

    StructField("source_tender_type", StringType(), True),
    StructField("tender_type", StringType(), True),

    StructField("source_status", StringType(), True),
    StructField("status", StringType(), True),

    StructField("publish_date", DateType(), True),
    StructField("closing_date", DateType(), True),
    StructField("opening_date", DateType(), True),

    StructField("categories", ArrayType(StringType()), True),
    StructField("regions", ArrayType(StringType()), True),

    StructField("estimated_value_min", DecimalType(18, 2), True),
    StructField("estimated_value_max", DecimalType(18, 2), True),
    StructField("document_price", DecimalType(18, 2), True),
    StructField("currency", StringType(), True),

    StructField("tender_url", StringType(), True),
    StructField("is_tech", BooleanType(), True),

    StructField("ingestion_timestamp", TimestampType(), True)
])

In [0]:
# NUPCO does not provide every field required by our common Silver schema. This cell maps the cleaned NUPCO data to the complete 24-column schema.

from pyspark.sql.functions import col, lit

nupco_silver_final = (
    nupco_silver

    # Identification
    .withColumn("reference_number", lit(None).cast("string"))

    # Tender information not provided by NUPCO
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("description", lit(None).cast("string"))
    .withColumn("agency_name", lit(None).cast("string"))

    # Preserve original NUPCO type
    .withColumn("source_tender_type", col("tender_type"))

    # Status
    .withColumn("source_status", lit(None).cast("string"))
    .withColumn("status", lit(None).cast("string"))

    # Dates
    .withColumn("publish_date", lit(None).cast("date"))

    # Classification
    .withColumn("categories", lit(None).cast("array<string>"))
    .withColumn("regions", lit(None).cast("array<string>"))

    # Financial information
    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn("currency", lit(None).cast("string"))

    # Furas classification
    .withColumn("is_tech", lit(None).cast("boolean"))

    # Put columns in canonical Silver order
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
nupco_silver_final.printSchema()

In [0]:
print("Rows:", nupco_silver_final.count())
print("Columns:", len(nupco_silver_final.columns))

In [0]:
display(nupco_silver_final)

In [0]:
# After mapping NUPCO to the common Silver schema, this cell performs a final completeness check on the most important NUPCO fields.

from pyspark.sql.functions import col, sum as spark_sum

important_columns = [
    "tender_key",
    "source",
    "source_tender_id",
    "tender_name",
    "closing_date",
    "tender_url",
    "ingestion_timestamp"
]

nupco_silver_final.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in important_columns
]).show()

In [0]:
# Check for Duplicate NUPCO Tenders

nupco_silver_final \
    .groupBy("source", "source_tender_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
# This cell is the final NUPCO loading step. It sends the standardized NUPCO dataset to its Silver Delta table.
# Our reusable merge function updates existing tenders using their unique tender key and inserts new tenders. On the first run, it creates the Delta table #automatically.

silver_nupco_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/nupco"
)

merge_to_silver(
    nupco_silver_final,
    silver_nupco_path,
    "NUPCO"
)

In [0]:
# After merging NUPCO into Silver, this cell reads the stored Delta table back from ADLS and verifies its row count and number of columns.
# This confirms that the Silver write was successful and that the stored dataset follows our 24-column schema.

nupco_silver_check = (
    spark.read
    .format("delta")
    .load(silver_nupco_path)
)

print("Silver rows:", nupco_silver_check.count())
print("Silver columns:", len(nupco_silver_check.columns))

display(nupco_silver_check)

# Source NO.2 : STC Bronze → Silver

In [0]:
# Read the stc path in the bronze layer

stc_path = bronze_path + "stc/"
display(dbutils.fs.ls(stc_path))

In [0]:
# checking and use the latest stc folder 

stc_folders = dbutils.fs.ls(stc_path)

latest_stc_folder = sorted(
    [f.path for f in stc_folders if f.isDir()],
    reverse=True
)[0]

print("Latest STC folder:", latest_stc_folder)

display(dbutils.fs.ls(latest_stc_folder))

In [0]:

stc_files = [
    f for f in dbutils.fs.ls(latest_stc_folder)
    if not f.isDir() and f.name.endswith(".json")
]


latest_stc_file = max(
    stc_files,
    key=lambda f: f.modificationTime
)

stc_file = latest_stc_file.path

print("Latest STC file:", stc_file)
stc_raw = (
    spark.read
    .option("multiline", "true")
    .json(stc_file)
)

stc_raw.printSchema()

# new cell here:
# ============================================================
# STEP: Load the latest STC Bronze file
# Purpose:
# Find the newest JSON file in the latest STC ingestion folder
# and load it into Spark for transformation.


In [0]:
display(stc_raw)

In [0]:
# This cell checks how many top-level records Spark loaded from the STC JSON file.
# Since the STC data may contain nested tender records, this is an initial loading check rather than the final tender count.

print("Top-level rows:", stc_raw.count())

In [0]:
# The STC Bronze JSON stores tenders inside a nested data array.
# This cell explodes that array so each tender becomes one row, extracts the relevant tender fields into normal Spark columns.
# preserves the source and ingestion metadata, and then checks the number of extracted tenders.

from pyspark.sql.functions import explode, col

stc_flat = (
    stc_raw
    .select(
        explode("data").alias("tender"),
        col("source"),
        col("ingestion_timestamp")
    )
)

stc_flat = stc_flat.select(
    col("tender.agency_name").alias("agency_name"),
    col("tender.confidence").alias("confidence"),
    col("tender.criticality_level").alias("criticality_level"),
    col("tender.details_url").alias("tender_url"),
    col("tender.portfolio_domain").alias("portfolio_domain"),
    col("tender.portfolio_fit_level").alias("portfolio_fit_level"),
    col("tender.publish_date").alias("publish_date"),
    col("tender.reason").alias("reason"),
    col("tender.reference_number").alias("reference_number"),
    col("tender.status").alias("status"),
    col("tender.status_ar").alias("status_ar"),
    col("tender.stc_portfolio_company").alias("stc_portfolio_company"),
    col("tender.stc_product").alias("stc_product"),
    col("tender.tender_activity_name").alias("tender_activity_name"),
    col("tender.tender_name").alias("tender_name"),
    col("tender.tender_type_name").alias("tender_type_name"),
    col("source"),
    col("ingestion_timestamp")
)

print("Number of STC tenders:", stc_flat.count())

display(stc_flat)

In [0]:
stc_flat.select(
    "publish_date",
    "status",
    "status_ar",
    "tender_type_name",
    "ingestion_timestamp"
).show(10, truncate=False)

In [0]:
# Data quality checks for STC tenders

from pyspark.sql.functions import sum as spark_sum

important_stc_columns = [
    "reference_number",
    "tender_name",
    "agency_name",
    "tender_type_name",
    "status",
    "publish_date",
    "tender_url",
    "ingestion_timestamp"
]

stc_flat.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in important_stc_columns
]).show()

In [0]:
print("Total:", stc_flat.count())

print(
    "Unique reference numbers:",
    stc_flat.select("reference_number").distinct().count()
)

print(
    "Null reference numbers:",
    stc_flat.filter(col("reference_number").isNull()).count()
)

In [0]:
# STC transformation and standardization cell
# We clean important fields, preserve the original STC status and tender type, standardize values such as status and dates, use the STC activity as a category,
# generate a unique tender key, and assign typed NULL values for information STC doesn't provide. This makes STC compatible with the other tender sources.

from pyspark.sql.functions import upper
from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    to_timestamp,
    lit,
    sha2,
    concat_ws,
    array
)

stc_silver = (
    stc_flat

    # Clean important text fields
    .withColumn("source_tender_id", trim(col("reference_number")))
    .withColumn("reference_number", trim(col("reference_number")))
    .withColumn("tender_name", trim(col("tender_name")))
    .withColumn("agency_name", trim(col("agency_name")))

    # Preserve original STC values
    .withColumn(
        "source_tender_type",
        trim(col("tender_type_name"))
    )
    .withColumn(
        "source_status",
        trim(col("status"))
    )

    # Standardized values
    # STC status is already simple: open -> OPEN
    .withColumn(
        "status",
        trim(col("status")).cast("string")
    )
    .withColumn(
        "status",
        upper(col("status"))
    )

    # For now preserve tender type as-is.
    # We can standardize Arabic tender types across ALL sources later.
    .withColumn(
        "tender_type",
        trim(col("tender_type_name"))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(col("publish_date"), "yyyy-MM-dd")
    )
    .withColumn(
        "ingestion_timestamp",
        to_timestamp(col("ingestion_timestamp"))
    )

    # Standard source
    .withColumn("source", lit("stc"))

    # Unique Furas tender key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Fields STC doesn't provide
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("description", lit(None).cast("string"))
    .withColumn("closing_date", lit(None).cast("date"))
    .withColumn("opening_date", lit(None).cast("date"))

    # STC activity can become a category
    .withColumn(
        "categories",
        array(trim(col("tender_activity_name")))
    )

    .withColumn("regions", lit(None).cast("array<string>"))

    # Financial fields unavailable
    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn("currency", lit(None).cast("string"))

    # We are not assigning the Furas tech flag yet
    .withColumn("is_tech", lit(None).cast("boolean"))

    # Exact 24-column Silver order
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# print and checking the schema for stc

stc_silver.printSchema()

print("Rows:", stc_silver.count())
print("Columns:", len(stc_silver.columns))

print(
    "Unique tender keys:",
    stc_silver.select("tender_key").distinct().count()
)

In [0]:
stc_silver.select(
    "source_tender_id",
    "tender_name",
    "source_tender_type",
    "tender_type",
    "source_status",
    "status",
    "publish_date",
    "categories"
).show(10, truncate=False)

In [0]:
# This cell loads the standardized STC dataset into its Silver Delta table. We use the same reusable merge function.
# where existing tenders are updated using tender_key and new tenders are inserted. This allows incremental processing instead of overwriting the entire source table.

silver_stc_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/stc"
)

merge_to_silver(
    stc_silver,
    silver_stc_path,
    "STC"
)

In [0]:
stc_silver_check = (
    spark.read
    .format("delta")
    .load(silver_stc_path)
)

print("Silver rows:", stc_silver_check.count())
print("Silver columns:", len(stc_silver_check.columns))
print(
    "Unique tender keys:",
    stc_silver_check.select("tender_key").distinct().count()
)

display(stc_silver_check)

# Etimad Bronze → Silver

In [0]:
# checking for etimad data path

etimad_path = bronze_path + "ETIMAD/"

display(dbutils.fs.ls(etimad_path))

In [0]:

# Get all date folders under ETIMAD
etimad_folders = dbutils.fs.ls(etimad_path)

# Keep folders only and choose the latest one
latest_etimad_folder = sorted(
    [f.path for f in etimad_folders if f.isDir()],
    reverse=True
)[0]

print("Latest ETIMAD folder:", latest_etimad_folder)

display(dbutils.fs.ls(latest_etimad_folder))

In [0]:
# # Get JSON files from the latest ETIMAD folder
# etimad_files = [
#     f for f in dbutils.fs.ls(latest_etimad_folder)
#     if not f.isDir() and f.name.endswith(".json")
# ]

# # Select the latest JSON file
# latest_etimad_file = max(
#     etimad_files,
#     key=lambda f: f.modificationTime
# )

# etimad_file = latest_etimad_file.path

# print("Latest ETIMAD file:", etimad_file)

# etimad_raw = (
#     spark.read
#     .option("multiline", "true")
#     .json(etimad_file)
# )

# etimad_raw.printSchema()

# print("Top-level rows:", etimad_raw.count())


# new cell here:
# ============================================================
# STEP: Load the latest ETIMAD Bronze file
# Purpose:
# Find the newest JSON file in the latest ETIMAD ingestion
# folder and load it into Spark for transformation.
# ============================================================

# Get JSON files from the latest ETIMAD folder
etimad_files = [
    f for f in dbutils.fs.ls(latest_etimad_folder)
    if not f.isDir() and f.name.endswith(".json")
]

# Stop if no ETIMAD JSON files are available
if not etimad_files:
    raise ValueError(
        f"No JSON files found in ETIMAD folder: {latest_etimad_folder}"
    )

# Select the most recently modified JSON file
latest_etimad_file = max(
    etimad_files,
    key=lambda f: f.modificationTime
)

etimad_file = latest_etimad_file.path

print("Latest ETIMAD file:", etimad_file)

# Read the raw ETIMAD JSON data
etimad_raw = (
    spark.read
    .option("multiline", "true")
    .json(etimad_file)
)

# Inspect the raw schema
etimad_raw.printSchema()

# Check the number of top-level records loaded
print("Top-level rows:", etimad_raw.count())

In [0]:
display(etimad_raw)

In [0]:
# checking the real etimad colmns

print("Number of Etimad columns:", len(etimad_raw.columns))

for c in etimad_raw.columns:
    print(c)

In [0]:
print("Number of Etimad tenders:", etimad_raw.count())

In [0]:
# This cell inspects the important raw ETIMAD fields before transformation.
# It helps us verify identifiers, tender information, dates, activity, price, and URL values so that we can map them correctly into the common Silver schema.

etimad_raw.select(
    "tenderId",
    "tenderIdString",
    "referenceNumber",
    "tenderNumber",
    "tenderName",
    "agencyName",
    "tenderTypeName",
    "tenderStatusName",
    "submitionDate",
    "lastOfferPresentationDate",
    "offersOpeningDate",
    "tenderActivityName",
    "condetionalBookletPrice",
    "ugrpRfxUrl"
).show(10, truncate=False)

In [0]:
# ETIMAD provides multiple identifier fields, so this cell compares their uniqueness and checks for missing values. This helps us choose a reliable source 
# tender ID for generating the Furas tender key while keeping the tender reference number separately.

from pyspark.sql.functions import col

print("Total Etimad tenders:", etimad_raw.count())

print(
    "Unique tenderId:",
    etimad_raw.select("tenderId").distinct().count()
)

print(
    "Unique tenderIdString:",
    etimad_raw.select("tenderIdString").distinct().count()
)

print(
    "Unique referenceNumber:",
    etimad_raw.select("referenceNumber").distinct().count()
)

print(
    "Null tenderId:",
    etimad_raw.filter(col("tenderId").isNull()).count()
)

print(
    "Null referenceNumber:",
    etimad_raw.filter(col("referenceNumber").isNull()).count()
)

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    to_timestamp,
    lit,
    sha2,
    concat_ws,
    array,
    upper
)

etimad_silver = (
    etimad_raw

    # IDs
    .withColumn(
        "source_tender_id",
        col("tenderId").cast("string")
    )
    .withColumn(
        "reference_number",
        trim(col("referenceNumber"))
    )

    # Main tender information
    .withColumn(
        "tender_name",
        trim(col("tenderName"))
    )
    .withColumn(
        "agency_name",
        trim(col("agencyName"))
    )

    # Preserve original Etimad values
    .withColumn(
        "source_tender_type",
        trim(col("tenderTypeName"))
    )
    .withColumn(
        "source_status",
        trim(col("tenderStatusName"))
    )

    # For now, preserve type/status as provided by Etimad.
    # Cross-source standardization comes later.
    .withColumn(
        "tender_type",
        trim(col("tenderTypeName"))
    )
    .withColumn(
        "status",
        trim(col("tenderStatusName"))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(to_timestamp(col("submitionDate")))
    )
    .withColumn(
        "closing_date",
        to_date(to_timestamp(col("lastOfferPresentationDate")))
    )
    .withColumn(
        "opening_date",
        to_date(to_timestamp(col("offersOpeningDate")))
    )

    # Category
    .withColumn(
        "categories",
        array(trim(col("tenderActivityName")))
    )

    # Price
    .withColumn(
        "document_price",
        col("condetionalBookletPrice").cast("decimal(18,2)")
    )

    # Source
    .withColumn("source", lit("etimad"))

    # Deterministic Furas key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Fields not available / not mapped from this Bronze dataset
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("description", lit(None).cast("string"))
    .withColumn("regions", lit(None).cast("array<string>"))
    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn("currency", lit("SAR"))
    .withColumn("is_tech", lit(None).cast("boolean"))

    # URL is not available in the sample field we inspected
    .withColumn("tender_url", lit(None).cast("string"))

    # Bronze file doesn't show a separate ingestion timestamp,
    # so keep it NULL rather than inventing one.
    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Exact common Silver schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
etimad_silver.printSchema()

print("Rows:", etimad_silver.count())
print("Columns:", len(etimad_silver.columns))

print(
    "Unique tender keys:",
    etimad_silver.select("tender_key").distinct().count()
)

In [0]:
etimad_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "agency_name",
    "source_tender_type",
    "source_status",
    "publish_date",
    "closing_date",
    "opening_date",
    "categories",
    "document_price",
    "currency"
).show(10, truncate=False)

# # if we have a url then:
# etimad_silver.select(
#     "source_tender_id",
#     "reference_number",
#     "tender_name",
#     "agency_name",
#     "source_tender_type",
#     "source_status",
#     "publish_date",
#     "closing_date",
#     "opening_date",
#     "categories",
#     "document_price",
#     "currency",
#     "tender_url"
# ).show(10, truncate=False)

In [0]:
# Checking Data Quality for dates:

print(
    "Null publish dates:",
    etimad_silver.filter(col("publish_date").isNull()).count()
)

print(
    "Null closing dates:",
    etimad_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    etimad_silver.filter(col("opening_date").isNull()).count()
)

In [0]:
# This cell loads the standardized ETIMAD data into its source-specific Silver Delta table.

silver_etimad_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/etimad"
)

merge_to_silver(
    etimad_silver,
    silver_etimad_path,
    "Etimad"
)

In [0]:
# This is the post-write validation for ETIMAD. It confirms that the Delta MERGE actually produced a readable Silver table with the expected structure and unique keys.

etimad_silver_check = (
    spark.read
    .format("delta")
    .load(silver_etimad_path)
)

print("Silver rows:", etimad_silver_check.count())
print("Silver columns:", len(etimad_silver_check.columns))
print(
    "Unique tender keys:",
    etimad_silver_check.select("tender_key").distinct().count()
)

# Source NO.3 : CST Bronze → Silver

In [0]:
# Checking the cst folder in the bronze

cst_path = bronze_path + "cst/"

display(dbutils.fs.ls(cst_path))

In [0]:

# Get all date folders under CST
cst_folders = dbutils.fs.ls(cst_path)

# Keep folders only and choose the latest one
latest_cst_folder = sorted(
    [f.path for f in cst_folders if f.isDir()],
    reverse=True
)[0]

print("Latest CST folder:", latest_cst_folder)

display(dbutils.fs.ls(latest_cst_folder))

In [0]:

# Get CSV files from the latest CST folder
# ============================================================
# STEP: Load the latest CST Bronze file
# Purpose:
# Find the newest CSV file in the latest CST ingestion folder
# and load it into Spark for transformation.
# ============================================================

cst_files = [
    f for f in dbutils.fs.ls(latest_cst_folder)
    if not f.isDir() and (
        f.name.endswith(".csv") or
        f.name.endswith("csv")
    )
]

print("CST files found:", [f.name for f in cst_files])

latest_cst_file = max(
    cst_files,
    key=lambda f: f.modificationTime
)

cst_file = latest_cst_file.path

print("Latest CST file:", cst_file)

cst_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiline", "true")
    .option("escape", '"')
    .csv(cst_file)
)

cst_raw.printSchema()

print("Number of CST records:", cst_raw.count())



In [0]:
# Print the original CST columns

print("Number of CST columns:", len(cst_raw.columns))

for c in cst_raw.columns:
    print(c)

In [0]:
display(cst_raw.limit(10))

In [0]:
cst_raw.select(
    "tender_id",
    "title",
    "deadline",
    "note",
    "type",
    "source",
    "extracted_at"
).show(15, truncate=False)

In [0]:
# This is a data-quality check before transforming CST into the common Silver schema

from pyspark.sql.functions import col, sum as spark_sum

print("Total CST tenders:", cst_raw.count())

print(
    "Unique tender IDs:",
    cst_raw.select("tender_id").distinct().count()
)

print(
    "Null tender IDs:",
    cst_raw.filter(col("tender_id").isNull()).count()
)

cst_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "tender_id",
        "title",
        "deadline",
        "note",
        "type",
        "source",
        "extracted_at"
    ]
]).show()

In [0]:
# This cell transforms the raw CST data into our common 24-column Silver schema.
# We map CST's ID, title, note and status to standardized fields, handle multiple deadline formats, preserve the extraction timestamp, generate a unique tender 
# key, and add typed NULL values for fields CST doesn't provide.

from pyspark.sql.functions import expr

from pyspark.sql.functions import (
    col,
    trim,
    to_timestamp,
    to_date,
    lit,
    sha2,
    concat_ws,
    upper
)

cst_silver = (
    cst_raw

    # IDs
    .withColumn(
        "source_tender_id",
        trim(col("tender_id"))
    )
    .withColumn(
        "reference_number",
        trim(col("tender_id"))
    )

    # Main tender information
    .withColumn(
        "tender_name",
        trim(col("title"))
    )
    .withColumn(
        "description",
        trim(col("note"))
    )

    # CST 'type' describes current/archive status
    .withColumn(
        "source_status",
        trim(col("type"))
    )
    .withColumn(
        "status",
        upper(trim(col("type")))
    )

    # Parse deadline
  .withColumn(
    "closing_date",
    to_date(
        expr("""
            coalesce(
                try_to_timestamp(trim(deadline), 'dd/MM/yyyy h:mm a'),
                try_to_timestamp(trim(deadline), 'dd/MM/yyyy')
            )
        """)
    )
)

    # Source
    .withColumn(
        "source",
        lit("cst")
    )

    # Unique Furas key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Existing extraction time becomes ingestion timestamp
    .withColumn(
        "ingestion_timestamp",
        col("extracted_at").cast("timestamp")
    )

    # Fields CST doesn't provide
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("agency_name", lit("CST").cast("string"))

    .withColumn("source_tender_type", lit(None).cast("string"))
    .withColumn("tender_type", lit(None).cast("string"))

    .withColumn("publish_date", lit(None).cast("date"))
    .withColumn("opening_date", lit(None).cast("date"))

    .withColumn("categories", lit(None).cast("array<string>"))
    .withColumn("regions", lit(None).cast("array<string>"))

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit(None).cast("string"))
    .withColumn("tender_url", lit(None).cast("string"))
    .withColumn("is_tech", lit(None).cast("boolean"))

    # Exact common Silver order
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# checking the schema

cst_silver.printSchema()

print("Rows:", cst_silver.count())
print("Columns:", len(cst_silver.columns))

print(
    "Unique tender keys:",
    cst_silver.select("tender_key").distinct().count()
)

print(
    "Null closing dates:",
    cst_silver.filter(col("closing_date").isNull()).count()
)

In [0]:
# # This cell loads the standardized CST dataset into its Silver Delta table. The reusable merge function uses tender_key to update existing tenders and insert 
# new ones

silver_cst_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/cst"
)

merge_to_silver(
    cst_silver,
    silver_cst_path,
    "CST"
)

In [0]:
# After merging CST into Silver, this cell reads the stored Delta table back from ADLS and verifies the number of records, confirms that it has the expected 
# 24-column schema, and checks that the tender keys remain unique.

cst_silver_check = (
    spark.read
    .format("delta")
    .load(silver_cst_path)
)

print("Silver rows:", cst_silver_check.count())
print("Silver columns:", len(cst_silver_check.columns))
print(
    "Unique tender keys:",
    cst_silver_check.select("tender_key").distinct().count()
)

## Soruce NO.4: Tanafos Bronze -> Silver

In [0]:
# ============================================================
# STEP: Load the latest Tanafos Bronze file

tanafos_path = bronze_path + "Tanafos/"

# Get all date folders under Tanafos
tanafos_folders = dbutils.fs.ls(tanafos_path)

# Choose latest date folder
latest_tanafos_folder = sorted(
    [f.path for f in tanafos_folders if f.isDir()],
    reverse=True
)[0]

print("Latest Tanafos folder:", latest_tanafos_folder)

# Get JSON files inside latest folder
tanafos_files = [
    f.path
    for f in dbutils.fs.ls(latest_tanafos_folder)
    if f.path.lower().endswith(".json")
]

# Choose latest JSON file
tanafos_file = sorted(tanafos_files, reverse=True)[0]

print("Latest Tanafos file:", tanafos_file)

# Read Bronze exactly as produced by the old ingestion method
tanafos_raw = (
    spark.read
    .option("multiline", "true")
    .json(tanafos_file)
)

tanafos_raw.printSchema()

print("Top-level rows:", tanafos_raw.count())


In [0]:
# checking tanafos columns:

print("Number of columns:", len(tanafos_raw.columns))

for c in tanafos_raw.columns:
    print(c)


In [0]:

display(tanafos_raw.limit(10))

In [0]:
# This cell displays a sample of the important raw Tanafos fields so we can understand their values and structure before mapping them to the common Silver schema.

tanafos_raw.select(
    "id",
    "ref_number",
    "title",
    "entity_name",
    "bid_type_name",
    "bid_status_id",
    "is_open",
    "creation_date",
    "submission_deadline",
    "classifications",
    "matched_classifications",
    "regions",
    "price",
    "is_tech",
    "url"
).show(10, truncate=False)

In [0]:
# These cells inspect the important raw Tanafos fields and perform data-quality checks before transformation.
# We compare the total number of records with unique IDs and reference numbers, check for missing IDs, and count null values in important fields.
# This helps us verify that id is suitable for generating the unique tender key and understand which source fields may be incomplete.


tanafos_raw.select(
    "id",
    "ref_number",
    "title",
    "entity_name",
    "bid_type_name",
    "bid_status_id",
    "is_open",
    "creation_date",
    "submission_deadline",
    "classifications",
    "matched_classifications",
    "regions",
    "price",
    "is_tech",
    "url"
).show(10, truncate=False)



from pyspark.sql.functions import col, sum as spark_sum

print("Total Tanafos tenders:", tanafos_raw.count())

print(
    "Unique IDs:",
    tanafos_raw.select("id").distinct().count()
)

print(
    "Unique reference numbers:",
    tanafos_raw.select("ref_number").distinct().count()
)

print(
    "Null IDs:",
    tanafos_raw.filter(col("id").isNull()).count()
)

tanafos_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "id",
        "ref_number",
        "title",
        "entity_name",
        "bid_type_name",
        "is_open",
        "creation_date",
        "submission_deadline",
        "price",
        "is_tech",
        "url"
    ]
]).show()

In [0]:
# This cell transforms Tanafos into our common 24-column Silver schema.
# It maps the source ID, reference number, title, agency, tender type, dates, categories, regions, price, technology flag, and URL. It also converts the 
# is_open field into a standardized OPEN or CLOSED status, generates the unique Furas tender key, and adds typed NULL values for fields that Tanafos does not 
# provide.

from pyspark.sql.functions import (
    col,
    trim,
    to_timestamp,
    to_date,
    lit,
    sha2,
    concat_ws,
    when
)

tanafos_silver = (
    tanafos_raw

    # IDs
    .withColumn(
        "source_tender_id",
        col("id").cast("string")
    )
    .withColumn(
        "reference_number",
        trim(col("ref_number"))
    )

    # Main information
    .withColumn(
        "tender_name",
        trim(col("title"))
    )
    .withColumn(
        "agency_name",
        trim(col("entity_name"))
    )

    # Tender type
    .withColumn(
        "source_tender_type",
        trim(col("bid_type_name"))
    )
    .withColumn(
        "tender_type",
        trim(col("bid_type_name"))
    )

    # Preserve source status information
    .withColumn(
        "source_status",
        col("bid_status_id").cast("string")
    )

    # Human-readable standardized status
    .withColumn(
        "status",
        when(col("is_open") == True, lit("OPEN"))
        .otherwise(lit("CLOSED"))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(to_timestamp(col("creation_date")))
    )
    .withColumn(
        "closing_date",
        to_date(to_timestamp(col("submission_deadline")))
    )

    # Source
    .withColumn(
        "source",
        lit("tanafos")
    )

    # Furas unique key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Existing arrays map directly
    .withColumn(
        "categories",
        col("classifications")
    )
    .withColumn(
        "regions",
        col("regions")
    )

    # Price
    .withColumn(
        "document_price",
        col("price").cast("decimal(18,2)")
    )

    # Existing tech classification
    .withColumn(
        "is_tech",
        col("is_tech").cast("boolean")
    )

    # URL
    .withColumn(
        "tender_url",
        trim(col("url"))
    )

    # Fields unavailable from this source
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("description", lit(None).cast("string"))
    .withColumn("opening_date", lit(None).cast("date"))

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit("SAR"))

    # Bronze file doesn't contain ingestion timestamp
    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Exact 24-column Silver schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# checking tanafos schema and also checking data quality: 

tanafos_silver.printSchema()

print("Rows:", tanafos_silver.count())
print("Columns:", len(tanafos_silver.columns))

print(
    "Unique tender keys:",
    tanafos_silver.select("tender_key").distinct().count()
)

print(
    "Null closing dates:",
    tanafos_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Tech tenders:",
    tanafos_silver.filter(col("is_tech") == True).count()
)

In [0]:

tanafos_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "agency_name",
    "source_tender_type",
    "status",
    "publish_date",
    "closing_date",
    "categories",
    "regions",
    "document_price",
    "currency",
    "is_tech"
).show(10, truncate=False)

In [0]:

# from delta.tables import DeltaTable

# tanafos_silver_path = (
#     "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/tanafos"
# )

# # If Silver already exists -> MERGE
# if DeltaTable.isDeltaTable(spark, tanafos_silver_path):

#     target = DeltaTable.forPath(spark, tanafos_silver_path)

#     (
#         target.alias("old")
#         .merge(
#             tanafos_silver.alias("new"),
#             "old.tender_key = new.tender_key"
#         )
#         .whenMatchedUpdateAll()
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

#     print("Tanafos Silver MERGE completed successfully!")

# # First run only -> create Silver
# else:
#     (
#         tanafos_silver.write
#         .format("delta")
#         .mode("overwrite")
#         .save(tanafos_silver_path)
#     )

#     print("Tanafos Silver created successfully!")

# new cell here:

# ============================================================
# STEP: Merge Tanafos data into the Silver layer
# Purpose:
# Incrementally update existing Tanafos tenders and insert
# new tenders using the reusable Silver MERGE function.
# ============================================================

tanafos_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/tanafos"
)

merge_to_silver(
    tanafos_silver,
    tanafos_silver_path,
    "Tanafos"
)

In [0]:
# Data checking after the merge to silver:

tanafos_check = (
    spark.read
    .format("delta")
    .load(tanafos_silver_path)
)

print("Rows:", tanafos_check.count())
print("Columns:", len(tanafos_check.columns))
print(
    "Unique tender keys:",
    tanafos_check.select("tender_key").distinct().count()
)


## Source NO.5: TAQEEM Bronze -> Silver

In [0]:

taqeem_path = bronze_path + "TAQEEM/"

# Get all date folders
taqeem_folders = dbutils.fs.ls(taqeem_path)

# Choose latest folder
latest_taqeem_folder = sorted(
    [f.path for f in taqeem_folders if f.isDir()],
    reverse=True
)[0]

print("Latest TAQEEM folder:", latest_taqeem_folder)

# Get JSON files from latest folder
taqeem_files = [
    f for f in dbutils.fs.ls(latest_taqeem_folder)
    if not f.isDir() and f.name.lower().endswith(".json")
]

# Choose latest JSON file
latest_taqeem_file = max(
    taqeem_files,
    key=lambda f: f.modificationTime
)

taqeem_file = latest_taqeem_file.path

print("Latest TAQEEM file:", taqeem_file)

# Read latest file
taqeem_raw = (
    spark.read
    .option("multiline", "true")
    .json(taqeem_file)
)

taqeem_raw.printSchema()

print("Top-level rows:", taqeem_raw.count())
print("Number of columns:", len(taqeem_raw.columns))

for c in taqeem_raw.columns:
    print(c)



In [0]:
display(taqeem_raw.limit(10))

In [0]:
# This cell displays a sample of the important TAQEEM fields before transformation. It helps us understand the identifiers, tender information, status, dates, 
# technology classification, and URL values so they can be mapped correctly to the common Silver schema.

taqeem_raw.select(
    "tender_id",
    "tender_number",
    "title",
    "tender_type",
    "status",
    "publish_date",
    "purpose_summary",
    "is_tech",
    "matched_keywords",
    "detail_url"
).show(10, truncate=False)

In [0]:
# This is the raw data-quality validation
# It compares the total number of tenders with unique tender IDs and tender numbers, checks whether any tender IDs are missing, and counts null values in the 
# important source fields before transformation.

from pyspark.sql.functions import col, sum as spark_sum

print("Total TAQEEM tenders:", taqeem_raw.count())

print(
    "Unique tender IDs:",
    taqeem_raw.select("tender_id").distinct().count()
)

print(
    "Unique tender numbers:",
    taqeem_raw.select("tender_number").distinct().count()
)

print(
    "Null tender IDs:",
    taqeem_raw.filter(col("tender_id").isNull()).count()
)

taqeem_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "tender_id",
        "tender_number",
        "title",
        "tender_type",
        "status",
        "publish_date",
        "purpose_summary",
        "is_tech",
        "detail_url"
    ]
]).show()

In [0]:
taqeem_raw.select(
    "tender_id",
    "tender_number",
    "title",
    "tender_type",
    "status",
    "is_tech",
    "detail_url"
).show(10, truncate=False)

In [0]:
# This cell profiles the distinct status and tender-type values available in TAQEEM. We inspect the source terminology before deciding how these fields should 
# be standardized in the Silver layer.

taqeem_raw.select("status").distinct().show(truncate=False)
taqeem_raw.select("tender_type").distinct().show(truncate=False)

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lit,
    sha2,
    concat_ws
)

taqeem_silver = (
    taqeem_raw

    # IDs
    .withColumn(
        "source_tender_id",
        trim(col("tender_id"))
    )
    .withColumn(
        "reference_number",
        trim(col("tender_number"))
    )

    # Main information
    .withColumn(
        "tender_name",
        trim(col("title"))
    )

    # Type
    .withColumn(
        "source_tender_type",
        trim(col("tender_type"))
    )
    .withColumn(
        "tender_type",
        trim(col("tender_type"))
    )

    # Status
    .withColumn(
        "source_status",
        trim(col("status"))
    )
    .withColumn(
        "status",
        upper(trim(col("status")))
    )

    # Source
    .withColumn(
        "source",
        lit("taqeem")
    )

    # Unique key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # URL and technology flag
    .withColumn(
        "tender_url",
        trim(col("detail_url"))
    )
    .withColumn(
        "is_tech",
        col("is_tech").cast("boolean")
    )

    # Fields unavailable in this Bronze source
    .withColumn("tender_name_en", lit(None).cast("string"))
    # .withColumn("description", lit(None).cast("string"))
    .withColumn(
    "description",
    trim(col("purpose_summary"))
    )
    .withColumn("agency_name", lit("TAQEEM"))

    .withColumn("publish_date", lit(None).cast("date"))
    .withColumn("closing_date", lit(None).cast("date"))
    .withColumn("opening_date", lit(None).cast("date"))

    .withColumn(
        "categories",
        lit(None).cast("array<string>")
    )
    .withColumn(
        "regions",
        lit(None).cast("array<string>")
    )

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit(None).cast("string"))
    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Exact common Silver schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# This cell validates the transformed TAQEEM dataset before loading it into Silver.

taqeem_silver.printSchema()

print("Rows:", taqeem_silver.count())
print("Columns:", len(taqeem_silver.columns))

print(
    "Unique tender keys:",
    taqeem_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    taqeem_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Tech tenders:",
    taqeem_silver.filter(col("is_tech") == True).count()
)

taqeem_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "source_tender_type",
    "source_status",
    "status",
    "is_tech"
).show(truncate=False)

In [0]:
taqeem_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/taqeem"
)

merge_to_silver(
    taqeem_silver,
    taqeem_silver_path,
    "TAQEEM"
)

In [0]:
# Data Quality checks:

taqeem_check = (
    spark.read
    .format("delta")
    .load(taqeem_silver_path)
)

print("Rows:", taqeem_check.count())
print("Columns:", len(taqeem_check.columns))
print(
    "Unique tender keys:",
    taqeem_check.select("tender_key").distinct().count()
)

## Source NO.6: TAHAKOM Bronze -> Silver

In [0]:
tahakom_path = bronze_path + "TAHAKOM/"

# Get all date folders under TAHAKOM
tahakom_folders = dbutils.fs.ls(tahakom_path)

# Choose latest folder
latest_tahakom_folder = sorted(
    [f.path for f in tahakom_folders if f.isDir()],
    reverse=True
)[0]

print("Latest TAHAKOM folder:", latest_tahakom_folder)

# Get JSON files from latest folder
tahakom_files = [
    f for f in dbutils.fs.ls(latest_tahakom_folder)
    if not f.isDir() and f.name.lower().endswith(".json")
]

# Choose latest JSON file
latest_tahakom_file = max(
    tahakom_files,
    key=lambda f: f.modificationTime
)

tahakom_file = latest_tahakom_file.path

print("Latest TAHAKOM file:", tahakom_file)

# Read latest file
tahakom_raw = (
    spark.read
    .option("multiline", "true")
    .json(tahakom_file)
)

tahakom_raw.printSchema()

print("Top-level rows:", tahakom_raw.count())
print("Number of columns:", len(tahakom_raw.columns))

for c in tahakom_raw.columns:
    print(c)



In [0]:
display(tahakom_raw.limit(10))

In [0]:
# This cell displays the important raw TAHAKOM fields before transformation.

tahakom_raw.select(
    "id",
    "tender_number",
    "Title_arabic",
    "Title_english",
    "Tender_type",
    "Status",
    "Tender_issuing_date",
    "Tender_open_date",
    "Proposals_deadline",
    "Tender_details_arabic",
    "Tender_details_english"
).show(10, truncate=False)

In [0]:
# Data checking 

from pyspark.sql.functions import col, sum as spark_sum

print("Total TAHAKOM tenders:", tahakom_raw.count())

print(
    "Unique IDs:",
    tahakom_raw.select("id").distinct().count()
)

print(
    "Unique tender numbers:",
    tahakom_raw.select("tender_number").distinct().count()
)

print(
    "Null IDs:",
    tahakom_raw.filter(col("id").isNull()).count()
)

tahakom_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "id",
        "tender_number",
        "Title_arabic",
        "Title_english",
        "Tender_type",
        "Status",
        "Tender_issuing_date",
        "Tender_open_date",
        "Proposals_deadline"
    ]
]).show()

In [0]:
# This cell transforms TAHAKOM into our common 24-column Silver schema. It maps the source ID and reference number, preserves the Arabic and English tender 
# titles, standardizes the tender type and status, converts the issuing, opening, and deadline dates, generates the unique Furas tender key, and adds typed 
# NULL values for fields TAHAKOM does not provide.

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_date,
    lit,
    sha2,
    concat_ws
)

tahakom_silver = (
    tahakom_raw

    # IDs
    .withColumn(
        "source_tender_id",
        col("id").cast("string")
    )
    .withColumn(
        "reference_number",
        trim(col("tender_number"))
    )

    # Names and description
    .withColumn(
        "tender_name",
        trim(col("Title_arabic"))
    )
    .withColumn(
        "tender_name_en",
        trim(col("Title_english"))
    )
    .withColumn(
        "description",
        trim(col("Tender_details_arabic"))
    )

    # Agency
    .withColumn(
        "agency_name",
        lit("TAHAKOM")
    )

    # Tender type
    .withColumn(
        "source_tender_type",
        trim(col("Tender_type"))
    )
    .withColumn(
        "tender_type",
        trim(col("Tender_type"))
    )

    # Status
    .withColumn(
        "source_status",
        trim(col("Status"))
    )
    .withColumn(
        "status",
        upper(trim(col("Status")))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(col("Tender_issuing_date"), "yyyy-MM-dd")
    )
    .withColumn(
        "closing_date",
        to_date(col("Proposals_deadline"), "yyyy-MM-dd")
    )
    .withColumn(
        "opening_date",
        to_date(col("Tender_open_date"), "yyyy-MM-dd")
    )

    # Source
    .withColumn(
        "source",
        lit("tahakom")
    )

    # Deterministic key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Fields unavailable in this source
    .withColumn("categories", lit(None).cast("array<string>"))
    .withColumn("regions", lit(None).cast("array<string>"))

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit(None).cast("string"))
    .withColumn("tender_url", lit(None).cast("string"))
    .withColumn("is_tech", lit(None).cast("boolean"))

    # createdAt is available, so preserve it as ingestion timestamp
    .withColumn(
        "ingestion_timestamp",
        col("createdAt").cast("timestamp")
    )

    # Exact 24-column schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# Checking tahakom_silver schema and the data quality

tahakom_silver.printSchema()

print("Rows:", tahakom_silver.count())
print("Columns:", len(tahakom_silver.columns))

print(
    "Unique tender keys:",
    tahakom_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    tahakom_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Null publish dates:",
    tahakom_silver.filter(col("publish_date").isNull()).count()
)

print(
    "Null closing dates:",
    tahakom_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    tahakom_silver.filter(col("opening_date").isNull()).count()
)

In [0]:
tahakom_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "tender_name_en",
    "source_tender_type",
    "status",
    "publish_date",
    "closing_date",
    "opening_date"
).show(10, truncate=False)

In [0]:

tahakom_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/tahakom"
)

merge_to_silver(
    tahakom_silver,
    tahakom_silver_path,
    "TAHAKOM"
)


In [0]:
# This is the post-write validation for TAHAKOM.

tahakom_check = (
    spark.read
    .format("delta")
    .load(tahakom_silver_path)
)

print("Rows:", tahakom_check.count())
print("Columns:", len(tahakom_check.columns))
print(
    "Unique tender keys:",
    tahakom_check.select("tender_key").distinct().count()
)

## Source NO.7: FORSAH Bronze -> Silver

In [0]:
forsah_path = bronze_path + "FORSAH/"

# Get all date folders
forsah_folders = dbutils.fs.ls(forsah_path)

# Choose latest folder
latest_forsah_folder = sorted(
    [f.path for f in forsah_folders if f.isDir()],
    reverse=True
)[0]

print("Latest FORSAH folder:", latest_forsah_folder)

# Get JSON files from latest folder
forsah_files = [
    f for f in dbutils.fs.ls(latest_forsah_folder)
    if not f.isDir() and f.name.lower().endswith(".json")
]

latest_forsah_file = max(
    forsah_files,
    key=lambda f: f.modificationTime
)

forsah_file = latest_forsah_file.path

print("Latest FORSAH file:", forsah_file)

forsah_raw = (
    spark.read
    .option("multiline", "true")
    .json(forsah_file)
)

forsah_raw.printSchema()

print("Top-level rows:", forsah_raw.count())
print("Number of columns:", len(forsah_raw.columns))

for c in forsah_raw.columns:
    print(c)




In [0]:
display(forsah_raw.limit(10))

In [0]:
# This cell inspects the important raw FORSAH fields before transformation, including identifiers, tender type and status, dates, categories, cities, estimated 
# values, technology classification

forsah_raw.select(
    "id",
    "title",
    "type_key",
    "type_name_ar",
    "type_name_en",
    "status_key",
    "publish_date",
    "due_date",
    "extended_due_date",
    "close_date",
    "categories_ar",
    "cities",
    "value_min",
    "value_max",
    "is_tech",
    "url"
).show(10, truncate=False)

In [0]:
# checks the quality and completeness of the raw FORSAH data before transforming it into Silver.

from pyspark.sql.functions import col, sum as spark_sum

print("Total FORSAH tenders:", forsah_raw.count())

print(
    "Unique IDs:",
    forsah_raw.select("id").distinct().count()
)

print(
    "Null IDs:",
    forsah_raw.filter(col("id").isNull()).count()
)

forsah_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "id",
        "title",
        "type_name_ar",
        "status_key",
        "publish_date",
        "due_date",
        "extended_due_date",
        "close_date",
        "value_min",
        "value_max",
        "is_tech",
        "url"
    ]
]).show()

In [0]:
# This cell profiles the distinct status and tender-type values used by FORSAH. We inspect these source values before standardizing them into the common Silver schema.

forsah_raw.select("status_key").distinct().show(truncate=False)

forsah_raw.select(
    "type_key",
    "type_name_ar",
    "type_name_en"
).distinct().show(truncate=False)

In [0]:
# checking for the date

forsah_raw.filter(
    col("extended_due_date").isNotNull()
).select(
    "id",
    "status_key",
    "publish_date",
    "due_date",
    "extended_due_date",
    "close_date"
).show(20, truncate=False)

In [0]:
# This cell transforms the raw FORSAH data into our common 24-column Silver schema. It standardizes identifiers, tender type, status and dates, uses the 
# extended deadline when available, maps categories and regions, preserves estimated tender values and the technology flag, and generates a unique tender key.

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_date,
    to_timestamp,
    coalesce,
    lit,
    sha2,
    concat_ws
)

forsah_silver = (
    forsah_raw

    # IDs
    .withColumn(
        "source_tender_id",
        trim(col("id"))
    )
    .withColumn(
        "reference_number",
        lit(None).cast("string")
    )

    # Main information
    .withColumn(
        "tender_name",
        trim(col("title"))
    )
    .withColumn(
        "tender_name_en",
        lit(None).cast("string")
    )
    .withColumn(
        "description",
        lit(None).cast("string")
    )
    .withColumn(
        "agency_name",
        lit(None).cast("string")
    )

    # Tender type
    .withColumn(
        "source_tender_type",
        trim(col("type_name_ar"))
    )
    .withColumn(
        "tender_type",
        trim(col("type_name_ar"))
    )

    # Status
    .withColumn(
        "source_status",
        trim(col("status_key"))
    )
    .withColumn(
        "status",
        upper(trim(col("status_key")))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(col("publish_date"), "yyyy-MM-dd")
    )
    .withColumn(
        "closing_date",
        to_date(
            coalesce(
                to_timestamp(
                    col("extended_due_date"),
                    "yyyy-MM-dd HH:mm:ss"
                ),
                to_timestamp(
                    col("due_date"),
                    "yyyy-MM-dd HH:mm:ss"
                )
            )
        )
    )
    .withColumn(
        "opening_date",
        lit(None).cast("date")
    )

    # Categories and regions already match our schema
    .withColumn(
        "categories",
        col("categories_ar")
    )
    .withColumn(
        "regions",
        col("cities")
    )

    # Estimated tender value
    .withColumn(
        "estimated_value_min",
        col("value_min").cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        col("value_max").cast("decimal(18,2)")
    )

    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn(
        "currency",
        lit("SAR")
    )

    # URL + tech classification
    .withColumn(
        "tender_url",
        trim(col("url"))
    )
    .withColumn(
        "is_tech",
        col("is_tech").cast("boolean")
    )

    # Source
    .withColumn(
        "source",
        lit("forsah")
    )

    # Unique key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Exact common Silver schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# Checking data quality for forsah and print the schema

forsah_silver.printSchema()

print("Rows:", forsah_silver.count())
print("Columns:", len(forsah_silver.columns))

print(
    "Unique tender keys:",
    forsah_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    forsah_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Null publish dates:",
    forsah_silver.filter(col("publish_date").isNull()).count()
)

print(
    "Null closing dates:",
    forsah_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Tech tenders:",
    forsah_silver.filter(col("is_tech") == True).count()
)

In [0]:
forsah_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/forsah"
)

merge_to_silver(
    forsah_silver,
    forsah_silver_path,
    "FORSAH"
)


In [0]:
# Checking the data

forsah_check = (
    spark.read
    .format("delta")
    .load(forsah_silver_path)
)

print("Rows:", forsah_check.count())
print("Columns:", len(forsah_check.columns))
print(
    "Unique tender keys:",
    forsah_check.select("tender_key").distinct().count()
)

## Source NO.8: GEO Bronze -> Silver

In [0]:

# GEO Bronze -> Silver

geo_path = bronze_path + "GEO/"

# Get all date folders
geo_folders = dbutils.fs.ls(geo_path)

# Choose latest folder automatically
latest_geo_folder = sorted(
    [f.path for f in geo_folders if f.isDir()],
    reverse=True
)[0]

print("Latest GEO folder:", latest_geo_folder)

# Get CSV files from latest folder
geo_files = [
    f for f in dbutils.fs.ls(latest_geo_folder)
    if not f.isDir() and f.name.lower().endswith(".csv")
]

latest_geo_file = max(
    geo_files,
    key=lambda f: f.modificationTime
)

geo_file = latest_geo_file.path

print("Latest GEO file:", geo_file)

# Read GEO file
geo_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiline", "true")
    .option("escape", '"')
    .csv(geo_file)
)

geo_raw.printSchema()

print("Rows:", geo_raw.count())
print("Number of columns:", len(geo_raw.columns))




In [0]:
geo_raw.show(10, truncate=False)

In [0]:
geo_raw.select("source").show(truncate=False)

content = geo_raw.select("content").first()["content"]

print("Content length:", len(content))
print()
print(content[:5000])

In [0]:
# This cell counts repeated Arabic labels in the raw GEO webpage content.

print("رقم المنافسة count:", content.count("رقم المنافسة"))
print("الرقم المرجعي count:", content.count("الرقم المرجعي"))
print("رقم المنافسة: count:", content.count("رقم المنافسة:"))

In [0]:
# This cell parses the raw GEO webpage text into individual tender sections. A regular expression identifies the beginning of each tender based on the webpage 
# structure, then non-tender header and footer content is removed. Finally, we count and preview the extracted tender sections to verify that the parsing 
# worked correctly

import re

content = geo_raw.select("content").first()["content"]

# The tender section starts with 01, 02, 03 ... followed by the title,
# and then "1 الجهة الحكومية"
pattern = r'(?=\b(?:0[1-9]|1[0-9])\s+.+?\s+1\s+[​\s]*الجهة الحكومية)'

sections = re.split(pattern, content)

# Remove page header / footer pieces that don't contain a tender ID
tender_sections = [
    s.strip()
    for s in sections
    if (
        "رقم المنافسة" in s
        or "الرقم المرجعي" in s
    )
]

print("Tender sections found:", len(tender_sections))

for i, section in enumerate(tender_sections, 1):
    print("\n----------------")
    print("Tender", i)
    print(section[:400])

In [0]:
# This cell tests the extraction of a unique identifier from each GEO tender section.

for i, section in enumerate(tender_sections, 1):

    competition = re.search(
        r'رقم\s+المنافسة\s*:\s*([^\s]+)',
        section
    )

    reference = re.search(
        r'الرقم\s+المرجعي\s*:\s*([^\s]+)',
        section
    )

    tender_id = (
        competition.group(1)
        if competition
        else reference.group(1)
        if reference
        else None
    )

    print(i, "->", tender_id)

In [0]:
# This cell searches the complete GEO webpage content for all competition numbers and reference numbers. It counts and displays the extracted identifiers so we 
# can verify that our regular expressions are correctly detecting the tender IDs before building the structured records.

import re

competition_ids = re.findall(
    r'رقم\s+المنافسة\s*:\s*([^\s]+)',
    content
)

reference_ids = re.findall(
    r'الرقم\s+المرجعي\s*:\s*([^\s]+)',
    content
)

print("Competition IDs:", len(competition_ids))
for x in competition_ids:
    print(x)

print("\nReference IDs:", len(reference_ids))
for x in reference_ids:
    print(x)

print("\nTotal IDs:", len(competition_ids) + len(reference_ids))

In [0]:
# Data checking

all_ids = competition_ids + reference_ids

print("IDs missing from tender_sections:")

for tender_id in all_ids:
    found = any(tender_id in section for section in tender_sections)

    if not found:
        print("MISSING:", tender_id)

In [0]:
# ============================================================
# STEP: Create structured GEO tender records
# Purpose:
# Convert each parsed GEO tender section into a structured
# record with one tender ID and its complete source text.
# ============================================================

import re

# Match either type of tender identifier
id_pattern = (
    r'(?:رقم\s+المنافسة|الرقم\s+المرجعي)\s*:\s*([^\s]+)'
)

matches = list(re.finditer(id_pattern, content))

print("IDs found:", len(matches))

geo_records = []

for i, match in enumerate(matches):
    tender_id = match.group(1)

    # Start a little before the ID so we retain the title/agency
    if i == 0:
        start = 0
    else:
        previous_end = matches[i - 1].end()
        start = previous_end

    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)

    geo_records.append({
        "tender_id": tender_id,
        "text": content[start:end].strip()
    })

print("Records created:", len(geo_records))

for i, r in enumerate(geo_records, 1):
    print(i, "->", r["tender_id"])



In [0]:
for i, r in enumerate(geo_records, 1):
    print("\n==============================")
    print("RECORD:", i)
    print("ID:", r["tender_id"])
    print(r["text"][:500])

In [0]:
# ============================================================
# STEP: Parse structured fields from GEO tender sections
# Purpose:
# Extract the tender ID, name, description, document price,
# closing date, and opening date from each GEO tender section.
# # ============================================================

import re
from pyspark.sql import Row

# Normalize whitespace first
clean_content = re.sub(r'\s+', ' ', content).strip()

# Find all 11 tender IDs
id_pattern = r'(?:رقم\s+المنافسة|الرقم\s+المرجعي)\s*:\s*([^\s]+)'
matches = list(re.finditer(id_pattern, clean_content))

parsed_records = []

for i, match in enumerate(matches):

    tender_id = match.group(1)

    # Use midpoint between IDs to isolate each tender.
    # Enough text before the ID is needed because title/agency come before it.
    previous_pos = matches[i - 1].end() if i > 0 else 0
    next_pos = matches[i + 1].start() if i + 1 < len(matches) else len(clean_content)

    block = clean_content[previous_pos:next_pos]

    # -------------------------
    # Tender name
    # -------------------------
    # Usually directly before رقم المنافسة / الرقم المرجعي
    title_match = re.search(
        r'2\s+(?:المعلومات الأساسية|​المعلومات الأساسية)\s+(.+?)'
        r'(?:رقم\s+المنافسة|الرقم\s+المنافسة|الرقم\s+المرجعي)',
        block
    )

    title = title_match.group(1).strip() if title_match else None

    # Remove labels occasionally included in title
    if title:
        title = re.sub(
            r'^(?:اسم|إسم|إﺳﻢ)\s+المنافسة\s*:\s*',
            '',
            title
        ).strip()

    # -------------------------
    # Price
    # -------------------------
    price_match = re.search(
        r'قيمة\s+المنافسة\s+([\d,.]+)',
        block
    )

    price = (
        float(price_match.group(1).replace(",", ""))
        if price_match
        else None
    )

    # -------------------------
    # Description / purpose
    # -------------------------
    purpose_match = re.search(
        r'(?:الغاية|الغرض)\s+من\s+المنافسة\s+(.+?)'
        r'(?=\s+[45]\s+(?:اخر|آخر))',
        block
    )

    description = (
        purpose_match.group(1).strip()
        if purpose_match
        else None
    )

    # -------------------------
    # Closing date
    # -------------------------
    closing_match = re.search(
        r'(?:آخر|اخر)\s+موعد\s+(?:لإستلام|لتقديم)\s+العروض'
        r'.*?الموافق\s*:?\s*(\d{2}/\d{2}/\d{4})',
        block
    )

    closing_date = (
        closing_match.group(1)
        if closing_match
        else None
    )

    # -------------------------
    # Opening date
    # -------------------------
    opening_match = re.search(
        r'تاريخ(?:\s+و\s+وقت)?\s+فتح\s+العروض'
        r'.*?الموافق\s*:?\s*(\d{2}/\d{2}/\d{4})',
        block
    )

    opening_date = (
        opening_match.group(1)
        if opening_match
        else None
    )

    parsed_records.append(
        Row(
            source_tender_id=tender_id,
            tender_name=title,
            description=description,
            document_price=price,
            closing_date_raw=closing_date,
            opening_date_raw=opening_date
        )
    )

geo_parsed = spark.createDataFrame(parsed_records)

geo_parsed.show(20, truncate=False)



In [0]:
print("Rows:", geo_parsed.count())

geo_parsed.select(
    "source_tender_id",
    "tender_name",
    "document_price",
    "closing_date_raw",
    "opening_date_raw"
).show(20, truncate=False)

In [0]:
# This is a data-quality validation cell.


from pyspark.sql.functions import col

for c in [
    "source_tender_id",
    "tender_name",
    "description",
    "document_price",
    "closing_date_raw",
    "opening_date_raw"
]:
    print(
        c,
        geo_parsed.filter(col(c).isNull()).count()
    )

In [0]:
# This cell identifies the specific GEO tenders that have missing document prices, closing dates, or opening dates after parsing. It helps us investigate 
# whether the values are actually missing from the source or whether the regex extraction needs adjustment.

geo_parsed.filter(
    col("document_price").isNull() |
    col("closing_date_raw").isNull() |
    col("opening_date_raw").isNull()
).select(
    "source_tender_id",
    "tender_name",
    "document_price",
    "closing_date_raw",
    "opening_date_raw"
).show(truncate=False)

In [0]:
# checking the data

missing_date_ids = [
    r["source_tender_id"]
    for r in geo_parsed.filter(
        col("closing_date_raw").isNull() |
        col("opening_date_raw").isNull()
    ).select("source_tender_id").collect()
]

print(missing_date_ids)

for tender_id in missing_date_ids:
    for r in geo_records:
        if r["tender_id"] == tender_id:
            print("\nID:", tender_id)
            print(r["text"])

In [0]:
# Remove invisible Unicode characters that appear in the webpage
clean_content = re.sub(r'[\u200b\u200c\u200d\u2060\ufeff]', '', content)
clean_content = re.sub(r'\s+', ' ', clean_content).strip()

# Rebuild matches after cleaning
id_pattern = r'(?:رقم\s+المنافسة|الرقم\s+المرجعي)\s*:\s*([^\s]+)'
matches = list(re.finditer(id_pattern, clean_content))

print("IDs after cleaning:", len(matches))

In [0]:
# ============================================================
# STEP: Apply known GEO source correction
# Purpose:
# Restore the closing and opening dates for tender 241039017854.
# These dates were available in the Bronze source content but
# were not captured by the standard GEO regex parser.
# ============================================================

from pyspark.sql.functions import when, lit, col

geo_parsed = (
    geo_parsed
    .withColumn(
        "closing_date_raw",
        when(
            col("source_tender_id") == "241039017854",
            lit("12/11/2024")
        ).otherwise(col("closing_date_raw"))
    )
    .withColumn(
        "opening_date_raw",
        when(
            col("source_tender_id") == "241039017854",
            lit("12/11/2024")
        ).otherwise(col("opening_date_raw"))
    )
)

In [0]:
for c in [
    "source_tender_id",
    "tender_name",
    "description",
    "document_price",
    "closing_date_raw",
    "opening_date_raw"
]:
    print(
        c,
        geo_parsed.filter(col(c).isNull()).count()
    )

In [0]:
# This cell verifies the known GEO date correction by displaying the affected tender after the update and confirming that its closing and opening dates were populated correctly.

geo_parsed.filter(
    col("source_tender_id") == "241039017854"
).show(truncate=False)

In [0]:
# This cell converts the parsed GEO tenders into our common 24-column Silver schema.

from pyspark.sql.functions import (
    col, lit, trim, to_date, sha2, concat_ws
)

geo_silver = (
    geo_parsed

    .withColumn(
        "source",
        lit("geo_resources")
    )

    # ID / reference number
    .withColumn(
        "reference_number",
        col("source_tender_id")
    )

    # No English title available
    .withColumn(
        "tender_name_en",
        lit(None).cast("string")
    )

    # Agency is the same for this source
    .withColumn(
        "agency_name",
        lit("الهيئة العامة للمساحة والمعلومات الجيومكانية")
    )

    # Tender type is not explicitly provided
    .withColumn(
        "source_tender_type",
        lit(None).cast("string")
    )
    .withColumn(
        "tender_type",
        lit(None).cast("string")
    )

    # Status is not explicitly provided
    .withColumn(
        "source_status",
        lit(None).cast("string")
    )
    .withColumn(
        "status",
        lit(None).cast("string")
    )

    # No publication date available
    .withColumn(
        "publish_date",
        lit(None).cast("date")
    )

    # Convert DD/MM/YYYY
    .withColumn(
        "closing_date",
        to_date(col("closing_date_raw"), "dd/MM/yyyy")
    )
    .withColumn(
        "opening_date",
        to_date(col("opening_date_raw"), "dd/MM/yyyy")
    )

    # No structured categories/regions extracted
    .withColumn(
        "categories",
        lit(None).cast("array<string>")
    )
    .withColumn(
        "regions",
        lit(None).cast("array<string>")
    )

    # No estimated contract value
    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )

    # Competition/document price
    .withColumn(
        "document_price",
        col("document_price").cast("decimal(18,2)")
    )

    .withColumn(
        "currency",
        lit("SAR")
    )

    # We don't have individual URLs
    .withColumn(
        "tender_url",
        lit(None).cast("string")
    )

    # Source doesn't explicitly classify records as tech
    .withColumn(
        "is_tech",
        lit(None).cast("boolean")
    )

    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Deterministic key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws(
                "|",
                col("source"),
                col("source_tender_id")
            ),
            256
        )
    )

    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# Data quality check

geo_silver.printSchema()

print("Rows:", geo_silver.count())
print("Columns:", len(geo_silver.columns))

print(
    "Unique tender keys:",
    geo_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    geo_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Null closing dates:",
    geo_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    geo_silver.filter(col("opening_date").isNull()).count()
)

print(
    "Null document prices:",
    geo_silver.filter(col("document_price").isNull()).count()
)

In [0]:
geo_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders/geo_resources"
)

merge_to_silver(
    geo_silver,
    geo_silver_path,
    "Geo Resources"
)


In [0]:
# Validate the Saved GEO Silver Table

geo_check = (
    spark.read
    .format("delta")
    .load(geo_silver_path)
)

print("Rows:", geo_check.count())
print("Columns:", len(geo_check.columns))
print(
    "Unique tender keys:",
    geo_check.select("tender_key").distinct().count()
)

geo_check.select(
    "source_tender_id",
    "tender_name",
    "closing_date",
    "opening_date",
    "document_price"
).show(20, truncate=False)

## Source NO. 10: SAR Bronze -> Silver

In [0]:

# SAR Bronze -> Silver
# ============================================================
# STEP: Load the latest SAR Bronze file
# Purpose:
# Find the latest SAR ingestion folder and newest CSV file,
# then load and inspect the raw data.
# ============================================================

sar_path = bronze_path + "SAR/"

# Get all date folders under SAR
sar_folders = dbutils.fs.ls(sar_path)

# Choose latest folder
latest_sar_folder = sorted(
    [f.path for f in sar_folders if f.isDir()],
    reverse=True
)[0]

print("Latest SAR folder:", latest_sar_folder)

# Get CSV files from latest folder
sar_files = [
    f for f in dbutils.fs.ls(latest_sar_folder)
    if not f.isDir() and f.name.lower().endswith(".csv")
]

# Choose latest CSV file
latest_sar_file = max(
    sar_files,
    key=lambda f: f.modificationTime
)

sar_file = latest_sar_file.path

print("Latest SAR file:", sar_file)

# Read latest SAR CSV
sar_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiline", "true")
    .option("escape", '"')
    .csv(sar_file)
)

sar_raw.printSchema()

print("Rows:", sar_raw.count())
print("Number of columns:", len(sar_raw.columns))

for c in sar_raw.columns:
    print(c)



In [0]:
# This cell profiles the raw SAR data before transformation.

sar_raw.show(10, truncate=False)

print("Total SAR tenders:", sar_raw.count())

print(
    "Unique tender IDs:",
    sar_raw.select("tender_id").distinct().count()
)

print(
    "Unique tender numbers:",
    sar_raw.select("tender_number").distinct().count()
)

for c in [
    "tender_id",
    "tender_number",
    "title",
    "description",
    "published_date",
    "closing_date",
    "source"
]:
    print(
        c,
        sar_raw.filter(col(c).isNull()).count()
    )

In [0]:
sar_raw.select(
    "tender_id",
    "tender_number",
    "published_date",
    "closing_date"
).show(10, truncate=False)

In [0]:
# This cell transforms the raw SAR tender data into our common Silver schema

from pyspark.sql.functions import (
    col, trim, upper, to_date, lit, sha2, concat_ws
)

sar_silver = (
    sar_raw

    .withColumn("source_tender_id", col("tender_id").cast("string"))
    .withColumn("reference_number", col("tender_number").cast("string"))

    .withColumn("tender_name", trim(col("title")))
    .withColumn("tender_name_en", lit(None).cast("string"))

    # description already exists
    .withColumn("description", trim(col("description")))

    # SAR = Saudi Arabia Railways
    .withColumn("agency_name", lit("Saudi Arabia Railways (SAR)"))

    .withColumn("source_tender_type", lit(None).cast("string"))
    .withColumn("tender_type", lit(None).cast("string"))

    .withColumn("source_status", lit(None).cast("string"))
    .withColumn("status", lit(None).cast("string"))

    # Dates: 17-Aug-2026
    .withColumn(
        "publish_date",
        to_date(col("published_date"), "dd-MMM-yyyy")
    )
    .withColumn(
        "closing_date",
        to_date(col("closing_date"), "dd-MMM-yyyy")
    )
    .withColumn(
        "opening_date",
        lit(None).cast("date")
    )

    .withColumn("categories", lit(None).cast("array<string>"))
    .withColumn("regions", lit(None).cast("array<string>"))

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit(None).cast("string"))
    .withColumn("tender_url", lit(None).cast("string"))
    .withColumn("is_tech", lit(None).cast("boolean"))
    .withColumn("ingestion_timestamp", lit(None).cast("timestamp"))

    # Normalize source name
    .withColumn("source", lit("sar"))

    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# Validate SAR Silver Transformation

print("Rows:", sar_silver.count())
print("Columns:", len(sar_silver.columns))

print(
    "Unique tender keys:",
    sar_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    sar_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Null publish dates:",
    sar_silver.filter(col("publish_date").isNull()).count()
)

print(
    "Null closing dates:",
    sar_silver.filter(col("closing_date").isNull()).count()
)

sar_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "publish_date",
    "closing_date"
).show(truncate=False)

In [0]:
sar_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders/sar"
)

merge_to_silver(
    sar_silver,
    sar_silver_path,
    "SAR"
)

In [0]:
# This cell reloads the SAR Silver Delta table after the merge and verifies that the persisted data has the expected 24-column schema and unique tender keys.

sar_check = (
    spark.read
    .format("delta")
    .load(sar_silver_path)
)

print("Rows:", sar_check.count())
print("Columns:", len(sar_check.columns))

print(
    "Unique tender keys:",
    sar_check.select("tender_key").distinct().count()
)

## tenders_unified

In [0]:
# Load All Source-Specific Silver Tables
# ============================================================
# STEP: Load all source-specific Silver Delta tables
# Purpose:
# Read each standardized tender source from the Silver layer
# and store the DataFrames for unified processing.
# ============================================================

silver_base = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/"
)

sources = [
    "nupco",
    "stc",
    "etimad",
    "cst",
    "tanafos",
    "taqeem",
    "tahakom",
    "forsah",
    "geo_resources",
    "sar"
]

silver_dfs = {}

for source in sources:
    path = silver_base + source

    df = spark.read.format("delta").load(path)
    silver_dfs[source] = df

    print(source, "->", df.count())

In [0]:
# ============================================================
# STEP: Validate Silver schema consistency
# Purpose:
# Confirm that every source-specific Silver dataset uses the
# same 24-column structure and column order before unification.
# ============================================================

expected_columns = silver_dfs["nupco"].columns

for source, df in silver_dfs.items():
    same_schema = df.columns == expected_columns

    print(
        source,
        "columns:", len(df.columns),
        "same schema:", same_schema
    )

In [0]:
# ============================================================
# STEP: Combine all source-specific Silver datasets
# Purpose:
# Union the standardized Silver DataFrames into one unified
# tender dataset before cross-source deduplication.
# ============================================================

from functools import reduce

tenders_unified = reduce(
    lambda df1, df2: df1.unionByName(df2),
    silver_dfs.values()
)

print("Unified rows:", tenders_unified.count())
print("Unified columns:", len(tenders_unified.columns))

tenders_unified.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()

In [0]:
# ============================================================
# STEP: Analyze duplicate reference numbers
# Purpose:
# Identify reference numbers that appear multiple times in the
# unified Silver data and determine whether they occur across
# multiple tender sources.
# ============================================================


from pyspark.sql.functions import (
    col, count, countDistinct
)

# How many records have a reference number?
print(
    "Rows with reference number:",
    tenders_unified.filter(
        col("reference_number").isNotNull()
    ).count()
)

print(
    "Distinct reference numbers:",
    tenders_unified.filter(
        col("reference_number").isNotNull()
    ).select("reference_number").distinct().count()
)

# Reference numbers appearing more than once
duplicate_refs = (
    tenders_unified
    .filter(col("reference_number").isNotNull())
    .groupBy("reference_number")
    .agg(
        count("*").alias("record_count"),
        countDistinct("source").alias("source_count")
    )
    .filter(col("record_count") > 1)
    .orderBy(col("record_count").desc())
)

print(
    "Duplicated reference numbers:",
    duplicate_refs.count()
)

duplicate_refs.show(30, truncate=False)

In [0]:
# ============================================================
# STEP: Identify cross-source duplicate references
# Purpose:
# Isolate reference numbers that appear in more than one
# tender source for further deduplication analysis.
# ============================================================


cross_source_duplicates = (
    duplicate_refs
    .filter(col("source_count") > 1)
)

print(
    "Cross-source duplicate references:",
    cross_source_duplicates.count()
)

cross_source_duplicates.show(30, truncate=False)

In [0]:
# ============================================================
# STEP: Analyze cross-source duplicate patterns
# Purpose:
# Identify which combinations of tender sources share the
# same reference numbers before defining deduplication rules.
# ============================================================


from pyspark.sql.functions import collect_set, size, sort_array

cross_source_detail = (
    tenders_unified
    .filter(col("reference_number").isNotNull())
    .groupBy("reference_number")
    .agg(
        count("*").alias("record_count"),
        sort_array(collect_set("source")).alias("sources")
    )
    .withColumn("source_count", size(col("sources")))
    .filter(col("source_count") > 1)
)

print(
    "Cross-source duplicate references:",
    cross_source_detail.count()
)

cross_source_detail.groupBy("sources") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(30, truncate=False)

In [0]:
# ============================================================
# STEP: Inspect cross-source duplicate tender records
# Purpose:
# Retrieve the actual tender records associated with duplicate
# reference numbers so their source, title, agency, dates,
# status, and URL can be compared before deduplication.
# ============================================================

duplicate_records = (
    tenders_unified.alias("t")
    .join(
        cross_source_detail.select("reference_number"),
        on="reference_number",
        how="inner"
    )
    .select(
        "reference_number",
        "source",
        "source_tender_id",
        "tender_name",
        "agency_name",
        "publish_date",
        "closing_date",
        "status",
        "tender_url"
    )
    .orderBy("reference_number", "source")
)

duplicate_records.show(40, truncate=False)

In [0]:
# ============================================================
# STEP: Analyze same-source duplicate references
# Purpose:
# Identify reference numbers that appear multiple times within
# the same source and determine whether those records have
# different source tender IDs.
# ============================================================

same_source_duplicates = (
    tenders_unified
    .filter(col("reference_number").isNotNull())
    .groupBy("source", "reference_number")
    .agg(
        count("*").alias("record_count"),
        countDistinct("source_tender_id").alias("tender_id_count")
    )
    .filter(col("record_count") > 1)
    .orderBy(col("record_count").desc())
)

print(
    "Same-source duplicate groups:",
    same_source_duplicates.count()
)

same_source_duplicates.show(50, truncate=False)

In [0]:
# ============================================================
# STEP: Inspect same-source duplicate tender records
# Purpose:
# Retrieve the actual records belonging to repeated reference
# numbers within the same source so their IDs, titles, dates,
# agencies, and statuses can be compared before deduplication.
# ============================================================

same_source_duplicate_records = (
    tenders_unified.alias("t")
    .join(
        same_source_duplicates.alias("d"),
        (col("t.source") == col("d.source")) &
        (col("t.reference_number") == col("d.reference_number")),
        "inner"
    )
    .select(
        col("t.source"),
        col("t.reference_number"),
        col("t.source_tender_id"),
        col("t.tender_name"),
        col("t.agency_name"),
        col("t.publish_date"),
        col("t.closing_date"),
        col("t.status")
    )
    .orderBy("source", "reference_number", "source_tender_id")
)

same_source_duplicate_records.show(50, truncate=False)

In [0]:
# ============================================================
# STEP: Compare STC and Etimad overlap completeness
# Purpose:
# Isolate tenders that appear in both STC and Etimad and
# compare the availability of important fields in each source
# before defining the cross-source merge strategy.
# ============================================================

stc_etimad_overlap = (
    tenders_unified
    .join(
        cross_source_detail
        .filter(
            (col("source_count") == 2) &
            (col("sources") == lit(["etimad", "stc"]))
        )
        .select("reference_number"),
        on="reference_number",
        how="inner"
    )
    .filter(col("source").isin("stc", "etimad"))
)

stc_etimad_overlap.groupBy("source").agg(
    count("*").alias("rows"),
    count("tender_name").alias("has_name"),
    count("agency_name").alias("has_agency"),
    count("publish_date").alias("has_publish_date"),
    count("closing_date").alias("has_closing_date"),
    count("opening_date").alias("has_opening_date"),
    count("categories").alias("has_categories"),
    count("tender_url").alias("has_url"),
    count("status").alias("has_status"),
    count("document_price").alias("has_document_price")
).show(truncate=False)

In [0]:
# ============================================================
# STEP: Merge overlapping STC and Etimad tenders
# Purpose:
# Create one canonical record for tenders found in both
# sources. Prefer Etimad values when available and use STC
# values as fallback to preserve complementary information.
# ============================================================


from pyspark.sql.functions import (
    col, coalesce, lit, sha2, concat_ws
)

stc_overlap = (
    stc_etimad_overlap
    .filter(col("source") == "stc")
    .alias("s")
)

etimad_overlap = (
    stc_etimad_overlap
    .filter(col("source") == "etimad")
    .alias("e")
)

merged_overlap = (
    etimad_overlap
    .join(
        stc_overlap,
        col("e.reference_number") == col("s.reference_number"),
        "inner"
    )

    # New canonical identity
    .select(
        sha2(
            concat_ws(
                "|",
                lit("etimad_stc"),
                col("e.reference_number")
            ),
            256
        ).alias("tender_key"),

        lit("etimad_stc").alias("source"),

        # Etimad ID is the canonical source ID
        col("e.source_tender_id").alias("source_tender_id"),
        col("e.reference_number").alias("reference_number"),

        # Core information
        coalesce(
            col("e.tender_name"),
            col("s.tender_name")
        ).alias("tender_name"),

        coalesce(
            col("e.tender_name_en"),
            col("s.tender_name_en")
        ).alias("tender_name_en"),

        coalesce(
            col("e.description"),
            col("s.description")
        ).alias("description"),

        coalesce(
            col("e.agency_name"),
            col("s.agency_name")
        ).alias("agency_name"),

        # Prefer Etimad tender type
        coalesce(
            col("e.source_tender_type"),
            col("s.source_tender_type")
        ).alias("source_tender_type"),

        coalesce(
            col("e.tender_type"),
            col("s.tender_type")
        ).alias("tender_type"),

        # Keep Etimad status for now
        coalesce(
            col("e.source_status"),
            col("s.source_status")
        ).alias("source_status"),

        coalesce(
            col("e.status"),
            col("s.status")
        ).alias("status"),

        # Dates
        coalesce(
            col("e.publish_date"),
            col("s.publish_date")
        ).alias("publish_date"),

        coalesce(
            col("e.closing_date"),
            col("s.closing_date")
        ).alias("closing_date"),

        coalesce(
            col("e.opening_date"),
            col("s.opening_date")
        ).alias("opening_date"),

        # Prefer Etimad categories
        coalesce(
            col("e.categories"),
            col("s.categories")
        ).alias("categories"),

        coalesce(
            col("e.regions"),
            col("s.regions")
        ).alias("regions"),

        coalesce(
            col("e.estimated_value_min"),
            col("s.estimated_value_min")
        ).alias("estimated_value_min"),

        coalesce(
            col("e.estimated_value_max"),
            col("s.estimated_value_max")
        ).alias("estimated_value_max"),

        # Etimad provides this
        coalesce(
            col("e.document_price"),
            col("s.document_price")
        ).alias("document_price"),

        coalesce(
            col("e.currency"),
            col("s.currency")
        ).alias("currency"),

        # STC provides the URL
        coalesce(
            col("e.tender_url"),
            col("s.tender_url")
        ).alias("tender_url"),

        coalesce(
            col("e.is_tech"),
            col("s.is_tech")
        ).alias("is_tech"),

        coalesce(
            col("e.ingestion_timestamp"),
            col("s.ingestion_timestamp")
        ).alias("ingestion_timestamp")
    )
)

In [0]:
# ============================================================
# STEP: Validate merged STC and Etimad records
# Purpose:
# Confirm that each overlapping reference produces one
# canonical record and check the completeness of important
# fields after combining information from both sources.
# ============================================================

print("Merged rows:", merged_overlap.count())
print("Columns:", len(merged_overlap.columns))

print(
    "Unique reference numbers:",
    merged_overlap.select("reference_number").distinct().count()
)

print(
    "Null URLs:",
    merged_overlap.filter(col("tender_url").isNull()).count()
)

print(
    "Null document prices:",
    merged_overlap.filter(col("document_price").isNull()).count()
)

print(
    "Null closing dates:",
    merged_overlap.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    merged_overlap.filter(col("opening_date").isNull()).count()
)

In [0]:
# ============================================================
# STEP: Prepare STC-Etimad overlap reference list
# Purpose:
# Create a list of reference numbers shared specifically by
# STC and Etimad so the original duplicate records can be
# replaced by their canonical merged versions.
# ============================================================


overlap_refs = (
    cross_source_detail
    .filter(
        (col("source_count") == 2) &
        (col("sources") == lit(["etimad", "stc"]))
    )
    .select("reference_number")
)

In [0]:
# ============================================================
# STEP: Remove original STC-Etimad overlap records
# Purpose:
# Keep all unified Silver records except the original tenders
# whose reference numbers belong to the confirmed STC-Etimad
# overlap. These records will be replaced by merged versions.
# ============================================================

non_overlap_records = (
    tenders_unified.alias("t")
    .join(
        overlap_refs.alias("o"),
        col("t.reference_number") == col("o.reference_number"),
        "left_anti"
    )
)

In [0]:
# ============================================================
# STEP: Build the deduplicated unified Silver dataset
# Purpose:
# Combine all non-overlapping tender records with the
# canonical merged STC-Etimad records to produce the final
# deduplicated unified Silver DataFrame.
# ============================================================

tenders_unified_dedup = (
    non_overlap_records
    .unionByName(merged_overlap)
)

In [0]:
# ============================================================
# STEP: Validate the deduplicated unified Silver dataset
# Purpose:
# Confirm the effect of the STC-Etimad merge and verify that
# the final unified dataset preserves the 24-column schema
# with unique and non-null tender keys.
# ============================================================

print("Before merge:", tenders_unified.count())
print("Non-overlap records:", non_overlap_records.count())
print("Merged records:", merged_overlap.count())
print("Final records:", tenders_unified_dedup.count())

print("Columns:", len(tenders_unified_dedup.columns))

print(
    "Unique tender keys:",
    tenders_unified_dedup.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    tenders_unified_dedup.filter(
        col("tender_key").isNull()
    ).count()
)

In [0]:
# ============================================================
# STEP: Validate final source distribution
# Purpose:
# Count the final unified Silver records by source after
# STC-Etimad deduplication and confirm that merged overlap
# records are represented by the new "etimad_stc" source.
# ============================================================

tenders_unified_dedup.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()
    

In [0]:
# remaining_cross_source_refs = (
#     tenders_unified_dedup
#     .filter(col("reference_number").isNotNull())
#     .groupBy("reference_number")
#     .agg(
#         countDistinct("source").alias("source_count")
#     )
#     .filter(col("source_count") > 1)
# )

# print(
#     "Remaining cross-source reference overlaps:",
#     remaining_cross_source_refs.count()
# )

# new cell here:

# ============================================================
# STEP: Check remaining cross-source reference overlaps
# Purpose:
# Verify whether any reference numbers still appear across
# multiple sources after the STC-Etimad deduplication.
# ============================================================

remaining_cross_source_refs = (
    tenders_unified_dedup
    .filter(col("reference_number").isNotNull())
    .groupBy("reference_number")
    .agg(
        countDistinct("source").alias("source_count")
    )
    .filter(col("source_count") > 1)
)

print(
    "Remaining cross-source reference overlaps:",
    remaining_cross_source_refs.count()
)

# Display remaining overlaps for investigation
remaining_cross_source_refs.show(30, truncate=False)

In [0]:
# ============================================================
# STEP: Profile tender status values across sources
# Purpose:
# Review the original source statuses and their current
# standardized status values before applying final status
# normalization rules to the unified Silver dataset.
# ============================================================

tenders_unified_dedup.groupBy(
    "source",
    "source_status",
    "status"
).count().orderBy(
    "source",
    col("count").desc()
).show(200, truncate=False)

In [0]:
# ============================================================
# STEP: Review overall tender status distribution
# Purpose:
# Identify all current status values in the unified Silver
# dataset and measure their frequency before applying final
# status normalization rules.
# ============================================================

tenders_unified_dedup.groupBy("status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(100, truncate=False)

In [0]:
# ============================================================
# STEP: Standardize tender status values
# Purpose:
# Map different source-specific status values into a common
# set of standardized statuses while preserving the original
# source value in the source_status column.
# ============================================================

from pyspark.sql.functions import (
    col, when, upper, trim, lit
)

tenders_status_clean = (
    tenders_unified_dedup
    .withColumn(
        "status",
        when(col("status").isNull(), lit("UNKNOWN"))

        .when(
            upper(trim(col("status"))).isin("OPEN", "CURRENT"),
            lit("OPEN")
        )

        .when(
            upper(trim(col("status"))).isin("CLOSED", "ENDED"),
            lit("CLOSED")
        )

        .when(
            upper(trim(col("status"))) == "PENDING",
            lit("PENDING")
        )

        .when(
            upper(trim(col("status"))) == "AWARDED",
            lit("AWARDED")
        )

        .when(
            upper(trim(col("status"))) == "COMPLETED",
            lit("COMPLETED")
        )

        .when(
            upper(trim(col("status"))).isin("CANCELED", "CANCELLED"),
            lit("CANCELLED")
        )

        .when(
            trim(col("status")) == "معتمدة",
            lit("APPROVED")
        )

        .when(
            upper(trim(col("status"))) == "RETURNED",
            lit("RETURNED")
        )

        .otherwise(lit("UNKNOWN"))
    )
)

In [0]:
# checking again

tenders_status_clean.groupBy("status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
print("Rows:", tenders_status_clean.count())
print("Columns:", len(tenders_status_clean.columns))
print(
    "Unique keys:",
    tenders_status_clean.select("tender_key").distinct().count()
)

In [0]:
# ============================================================
# STEP: Profile tender type values across sources
# Purpose:
# Compare the original source tender types with their current
# standardized tender_type values and identify differences
# that may require additional normalization.
# ============================================================

tenders_status_clean.groupBy(
    "source",
    "source_tender_type",
    "tender_type"
).count().orderBy(
    "source",
    col("count").desc()
).show(200, truncate=False)

In [0]:
tenders_status_clean.groupBy("tender_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(100, truncate=False)

In [0]:
# ============================================================
# STEP: Standardize tender type values
# Purpose:
# Map different Arabic and English source tender types into
# a consistent set of standardized tender type categories.
# Unmapped values are labeled UNKNOWN while the original
# value remains available in source_tender_type.
# ============================================================


from pyspark.sql.functions import col, when, trim, upper, lit

tenders_type_clean = (
    tenders_status_clean
    .withColumn(
        "tender_type",

        # Missing
        when(
            col("tender_type").isNull(),
            lit("UNKNOWN")
        )

        # Direct Purchase
        .when(
            trim(col("tender_type")).isin(
                "شراء مباشر",
                "Direct Purchase"
            ),
            lit("DIRECT_PURCHASE")
        )

        # Public Tender
        .when(
            trim(col("tender_type")).isin(
                "منافسة عامة",
                "مناقصة",
                "عامة",
                "Public tender"
            ),
            lit("PUBLIC_TENDER")
        )

        # Framework Agreement
        .when(
            trim(col("tender_type")) == "منافسة إتفاقية إطارية",
            lit("FRAMEWORK_AGREEMENT")
        )

        # Two-stage tender
        .when(
            trim(col("tender_type")) ==
            "المنافسة على مرحلتين (المرحلة الاولى)",
            lit("TWO_STAGE_TENDER")
        )

        # RFQ
        .when(
            upper(trim(col("tender_type"))) == "RFQ",
            lit("RFQ")
        )

        # RFP
        .when(
            upper(trim(col("tender_type"))) == "RFP",
            lit("RFP")
        )

        # Tanafos
        .when(
            trim(col("tender_type")) == "فورية",
            lit("IMMEDIATE")
        )

        .when(
            trim(col("tender_type")) == "محدودة",
            lit("LIMITED_TENDER")
        )

        # Other Etimad/STC types
        .when(
            trim(col("tender_type")) == "منافسات الشركات الكبرى",
            lit("MAJOR_COMPANIES")
        )

        .when(
            trim(col("tender_type")) == "المزايدة العكسية الالكترونية",
            lit("REVERSE_AUCTION")
        )

        .when(
            trim(col("tender_type")) == "المشاركة فى الدخل",
            lit("REVENUE_SHARING")
        )

        .when(
            trim(col("tender_type")) == "مسابقة",
            lit("COMPETITION")
        )

        # Anything else, including NUPCO lifecycle labels
        .otherwise(lit("UNKNOWN"))
    )
)

In [0]:
# checking the tender type

tenders_type_clean.groupBy("tender_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(50, truncate=False)

In [0]:
print("Rows:", tenders_type_clean.count())
print("Columns:", len(tenders_type_clean.columns))

print(
    "Unique keys:",
    tenders_type_clean.select("tender_key").distinct().count()
)

In [0]:
# ============================================================
# STEP: Check NULL values across the unified Silver dataset
# Purpose:
# Measure data completeness by counting NULL values in every
# column after deduplication and standardization.
# ===========================================================

from pyspark.sql.functions import col, count, when

quality_check = tenders_type_clean.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in tenders_type_clean.columns
])

quality_check.show(vertical=True)

In [0]:
# ============================================================
# STEP: Validate critical fields and date consistency
# Purpose:
# Verify that essential tender identifiers are present and
# detect invalid date sequences where closing or opening dates
# occur before the tender publication date.
# ============================================================

print(
    "Null tender_key:",
    tenders_type_clean.filter(col("tender_key").isNull()).count()
)

print(
    "Null source:",
    tenders_type_clean.filter(col("source").isNull()).count()
)

print(
    "Null source_tender_id:",
    tenders_type_clean.filter(col("source_tender_id").isNull()).count()
)

print(
    "Null tender_name:",
    tenders_type_clean.filter(col("tender_name").isNull()).count()
)

print(
    "Closing before publish:",
    tenders_type_clean.filter(
        col("closing_date") < col("publish_date")
    ).count()
)

print(
    "Opening before publish:",
    tenders_type_clean.filter(
        col("opening_date") < col("publish_date")
    ).count()
)

In [0]:
# check each source and it's render counts

tenders_type_clean.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()

In [0]:
# checking data quality and if there is incorrect values like opening date < publish date

tenders_type_clean.filter(
    col("opening_date") < col("publish_date")
).select(
    "source",
    "source_tender_id",
    "reference_number",
    "tender_name",
    "publish_date",
    "closing_date",
    "opening_date"
).show(truncate=False)

In [0]:
tenders_type_clean.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()

## final unified Silver Delta

In [0]:
# ============================================================
# STEP: Write the final unified Silver dataset
# Purpose:
# Persist the cleaned, standardized, and deduplicated unified
# tender dataset to ADLS as a Delta table.
#
# The unified table is rebuilt using overwrite because
# cross-source deduplication relationships may change when
# new source data arrives.
# ============================================================

final_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders_unified"
)

(
    tenders_type_clean
    .write
    .format("delta")
    .mode("overwrite")
    .save(final_silver_path)
)

print("Unified Silver dataset written successfully.")

In [0]:
# checking the silver final rows & columns and unique keys

silver_final = (
    spark.read
    .format("delta")
    .load(final_silver_path)
)

print("Rows:", silver_final.count())
print("Columns:", len(silver_final.columns))
print(
    "Unique keys:",
    silver_final.select("tender_key").distinct().count()
)

In [0]:
silver_final.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()

In [0]:
# ============================================================
# STEP: Validate all source-specific Silver Delta tables
# Purpose:
# Read each persisted Silver source table and verify that
# tender_key values are unique after incremental processing.
# ============================================================

silver_sources = {
    "NUPCO": silver_nupco_path,
    "STC": silver_stc_path,
    "Etimad": silver_etimad_path,
    "CST": silver_cst_path,
    "Tanafos": tanafos_silver_path,
    "TAQEEM": taqeem_silver_path,
    "TAHAKOM": tahakom_silver_path,
    "FORSAH": forsah_silver_path,
    "Geo Resources": geo_silver_path,
    "SAR": sar_silver_path
}

print("Silver validation")
print("-" * 55)

for source_name, path in silver_sources.items():
    df = spark.read.format("delta").load(path)

    total = df.count()
    unique_keys = df.select("tender_key").distinct().count()
    duplicates = total - unique_keys

    print(
        f"{source_name:15} "
        f"rows={total:<6} "
        f"unique={unique_keys:<6} "
        f"duplicates={duplicates}"
    )

In [0]:
# ============================================================
# STEP: Validate the persisted unified Silver Delta table
# Purpose:
# Read the final unified dataset back from ADLS and verify

unified_check = (
    spark.read
    .format("delta")
    .load(final_silver_path)
)

print("Unified rows:", unified_check.count())
print(
    "Unique tender keys:",
    unified_check.select("tender_key").distinct().count()
)

print(
    "Duplicate keys:",
    unified_check.count()
    - unified_check.select("tender_key").distinct().count()
)

## Checking before creating unity catalog

In [0]:
# ============================================================
# STEP: Inspect unified Silver tender dataset
# Purpose:
# Verify the existing unified Silver Delta dataset, its schema,
# and record count before registering it in Unity Catalog.
# ============================================================

unified_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders_unified"
)

# Read the existing unified Silver Delta dataset.
unified_silver = (
    spark.read
    .format("delta")
    .load(unified_silver_path)
)

# Verify the standardized schema.
unified_silver.printSchema()

# Check the current number of unified tender records.
print(
    "Unified Silver record count:",
    unified_silver.count()
)

# Preview records from the unified dataset.
display(unified_silver.limit(10))

In [0]:
# ============================================================
# STEP: Profile Silver data for Gold star-schema design
# Purpose:
# Understand the actual source, agency, category, and region
# values before creating Gold fact and dimension tables.
# ============================================================

silver_unified = spark.table(
    "tenderdatabricks.silver.tenders_unified"
)

# ------------------------------------------------------------
# 1. Sources
# Check the distinct sources and number of tenders per source.
# ------------------------------------------------------------

display(
    silver_unified
    .groupBy("source")
    .count()
    .orderBy("source")
)


# ------------------------------------------------------------
# 2. Agencies
# Check how many different agencies exist and how often
# agency_name is missing.
# ------------------------------------------------------------

silver_unified.select("agency_name").distinct().show(
    50,
    truncate=False
)

print(
    "Distinct agencies:",
    silver_unified.select("agency_name").distinct().count()
)

print(
    "Null agencies:",
    silver_unified.filter("agency_name IS NULL").count()
)


# ------------------------------------------------------------
# 3. Categories
# Inspect the array values currently stored in Silver.
# ------------------------------------------------------------

display(
    silver_unified
    .select("categories")
    .where("categories IS NOT NULL")
    .limit(20)
)


# ------------------------------------------------------------
# 4. Regions
# Inspect the array values currently stored in Silver.
# ------------------------------------------------------------

display(
    silver_unified
    .select("regions")
    .where("regions IS NOT NULL")
    .limit(20)
)